<a href="https://colab.research.google.com/github/Hamna-Riaz0375/flyrankAI-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML Task Framing: Client Churn Prediction

**Student:** Hamna Riaz
**Date:** 2026-08-01
**Lane:** Client Churn Prediction (Lane 1)

In [5]:
# Install libraries
!pip install pandas numpy -q

import pandas as pd
import numpy as np

print("✅ Libraries imported!")

# Load data from correct URL
url = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'

try:
    df = pd.read_csv(url)
    print(f"✅ Data loaded! Shape: {df.shape}")
    display(df.head())
except Exception as e:
    print(f"⚠️ Error: {e}")
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(list(uploaded.keys())[0])
    print(f"✅ Data uploaded! Shape: {df.shape}")
    display(df.head())

✅ Libraries imported!
✅ Data loaded! Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My Lane as an ML Task

**Lane:** Client Churn Prediction

**ML Task Type:** Binary Classification

### Why Binary Classification?

We are predicting one of **two outcomes**:
- Client churns (Yes / 1)
- Client stays (No / 0)

### Why Not Other Task Types?

| Task Type | Would It Work? | Why Not? |
|-----------|---------------|----------|
| **Regression** | Partially | Would predict probability (0-1), but business wants clear Yes/No |
| **Clustering** | No | We don't want to group clients, we want to predict behavior |
| **Ranking** | Could work | Ranking clients by risk is useful, but classification is primary task |
| **Multi-class** | No | Only two outcomes: churn or not churn |

### Conclusion:
Binary classification is the right choice because:
1. Business action is binary (contact or don't contact)
2. Interpretability is important for account managers
3. Evaluation is straightforward (confusion matrix, precision/recall)

## 2. Target or Proxy

### Primary Target:
**Client Churn** (Binary: 1 = Churned, 0 = Stayed)

### What is Churn?
- A client who stops using our services
- A client who doesn't renew their contract
- A client who hasn't engaged with content for 30+ days

### Proxy Variable (if churn not directly available):
- `last_engagement_date` < 30 days ago
- `content_consumption` dropped by 80%
- `support_tickets` increased by 50%

### How to Create Target Column:
```python
# Example: If we had client-level data
df['churned'] = df['last_engagement_days'] > 30

In [6]:
# Check if we have client_id column
if 'client_id' in df.columns:
    print("📊 Creating simulated churn target for demonstration")

    # Create a simulated target based on engagement
    client_agg = df.groupby('client_id').agg({
        'search_volume': 'mean',
        'cpc': 'mean',
        'competition': 'mean'
    }).reset_index()

    # Simulate churn: clients with low search_volume + high competition
    client_agg['simulated_churn'] = (
        (client_agg['search_volume'] < 10) &
        (client_agg['competition'] > 0.5)
    ).astype(int)

    print(f"📊 Client-level data shape: {client_agg.shape}")
    print("\n🔍 Sample with Target Column:")
    display(client_agg.head(10))

    print("\n📈 Churn Distribution:")
    print(client_agg['simulated_churn'].value_counts())
    print(f"\nChurn Rate: {client_agg['simulated_churn'].mean() * 100:.2f}%")

else:
    print("⚠️ 'client_id' column not found. Showing current data:")
    display(df.head(3))
    print("\nNote: Target column would be 'churned' in real data.")

📊 Creating simulated churn target for demonstration
📊 Client-level data shape: (32, 5)

🔍 Sample with Target Column:


,client_id,search_volume,cpc,competition,simulated_churn
0,client_02d20bbd7e,581.842105,0.895000,0.263158,0
1,client_0b918943df,657.894737,0.731053,0.620000,0
2,client_19581e27de,220.927010,0.585771,0.178207,0
3,client_1a6562590e,116.666667,0.200000,0.110000,0
4,client_25fc0e7096,NaN,NaN,NaN,0
5,client_2c624232cd,38.056537,0.048905,0.028463,0
6,client_349c41201b,71.979030,0.196225,0.219725,0
7,client_3fdba35f04,431.169462,1.191289,0.306152,0
8,client_434c9b5ae5,25.287356,1.943563,0.101724,0
9,client_4e07408562,285.465328,0.377509,0.129270,0



📈 Churn Distribution:
simulated_churn
0    32
Name: count, dtype: int64

Churn Rate: 0.00%


## 3. Success Metric

### Primary Metric: F1-Score

### Why F1-Score?

We need **balance** between Precision and Recall:
- **False Negatives** (missing churn) → Expensive (lost clients)
- **False Positives** (unnecessary action) → Costly (wasted effort)

### Comparison of Metrics:

| Metric | Good For | Why Not Primary |
|--------|----------|-----------------|
| **Accuracy** | Balanced classes | Churn is usually imbalanced (few churners) |
| **Precision** | Minimizing false alarms | We might miss actual churners |
| **Recall** | Finding all churners | Might waste resources on false alarms |
| **AUC-ROC** | Overall ranking ability | Less interpretable for business |
| **F1-Score** | Balance of both | ✅ **BEST CHOICE** |

### Business Threshold:
- Accounts with churn probability > 70% → Action required
- Prioritize **Recall** (find churners) over Precision

### Target Performance:
- **F1-Score > 0.70** → Model is useful
- **Recall > 0.75** → We find most churners
- **Precision > 0.60** → We don't waste too much effort

### Secondary Metrics:
- **Confusion Matrix** (for interpretability)
- **AUC-ROC** (for model comparison)
- **Business Impact:** Clients saved × Average Revenue

## 4. The Unit of Analysis

### Current Unit (Data we have):
**One row = One Content Piece**
- 30,000 rows × 44 columns
- Each row: content_id, client_id, search_volume, cpc, content_type, etc.

### Unit We Need (For Churn Prediction):
**One row = One Client**

### Why Client-Level Unit?
- Churn happens at **client level**, not content level
- We need to predict **client behavior**
- Features must be aggregated per client

### How to Aggregate to Client Level:

**Client Features to Create:**

| Feature Type | Example Features |
|--------------|------------------|
| **Engagement** | Total content pieces, avg search_volume, content diversity |
| **Performance** | Avg CPC, competition level mix, CTR, avg position |
| **Behavioral** | Recency of engagement, consistency, trends over time |

### Aggregation Code:
```python
client_df = df.groupby('client_id').agg({
    'search_volume': ['mean', 'max', 'count'],
    'cpc': 'mean',
    'competition': 'mean',
    'content_type': 'nunique'
}).reset_index()

In [7]:
print("="*60)
print("📊 CURRENT UNIT: One row = one content piece")
print("="*60)
print(f"Shape: {df.shape}")
print("\nSample rows (content level):")
display(df.head(3))

# Check if client_id exists
if 'client_id' in df.columns:
    print("\n" + "="*60)
    print("📊 FUTURE UNIT: One row = one client")
    print("="*60)

    # Aggregate to client level
    client_df = df.groupby('client_id').agg({
        'search_volume': ['mean', 'count'],
        'cpc': 'mean',
        'competition': 'mean',
        'word_count': 'mean'
    }).reset_index()

    # Flatten column names
    client_df.columns = ['client_id', 'avg_search_volume', 'content_count', 'avg_cpc', 'avg_competition', 'avg_word_count']

    # Simulate churn target
    client_df['churned'] = (
        (client_df['avg_search_volume'] < 10) &
        (client_df['avg_competition'] > 0.5)
    ).astype(int)

    print(f"Client-level shape: {client_df.shape}")
    print("\n🔍 Sample client-level data (one row = one client):")
    display(client_df.head(5))

    print("\n📊 Explanation:")
    print(f"- Total clients: {len(client_df)}")
    print(f"- Features: engagement (search_volume), activity (content_count), performance (CPC, competition)")
    print(f"- Target: churned (0 = stayed, 1 = churned)")
    print(f"- Churn rate: {client_df['churned'].mean() * 100:.2f}%")

else:
    print("\n⚠️ 'client_id' column not found.")
    print("Current unit: One row = one content piece")
    display(df.head(3))

📊 CURRENT UNIT: One row = one content piece
Shape: (30000, 44)

Sample rows (content level):


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9



📊 FUTURE UNIT: One row = one client
Client-level shape: (32, 7)

🔍 Sample client-level data (one row = one client):


,client_id,avg_search_volume,content_count,avg_cpc,avg_competition,avg_word_count,churned
0,client_02d20bbd7e,581.842105,38,0.895000,0.263158,2040.500000,0
1,client_0b918943df,657.894737,19,0.731053,0.620000,3184.800000,0
2,client_19581e27de,220.927010,7001,0.585771,0.178207,2964.792320,0
3,client_1a6562590e,116.666667,3,0.200000,0.110000,1539.000000,0
4,client_25fc0e7096,NaN,0,NaN,NaN,3644.077731,0



📊 Explanation:
- Total clients: 32
- Features: engagement (search_volume), activity (content_count), performance (CPC, competition)
- Target: churned (0 = stayed, 1 = churned)
- Churn rate: 0.00%


## 5. Why ML Beats a Fixed Rule Here

### ❌ Why a Simple Rule Won't Work:

**Example Rule:** "If search_volume < 10, then client will churn"

**Problems with This Rule:**

| Problem | Explanation |
|---------|-------------|
| **Too Simplistic** | Churn depends on many factors, not just one |
| **Context Matters** | Client with HIGH competition may churn even with search_volume = 50 |
| **Non-Linear** | Churn risk doesn't increase linearly |
| **Threshold Issues** | Where to set the rule? 10? 20? 50? |
| **No Adaptation** | Rule can't learn from new patterns |

---

### ✅ Why ML Works Better:

| ML Capability | How It Helps |
|---------------|--------------|
| **Multiple Factors** | Can consider 44+ features simultaneously |
| **Complex Patterns** | Can detect non-linear relationships |
| **Learns from Data** | Adapts to changing client behavior |
| **Probability Output** | Gives risk scores, not just Yes/No |
| **Continuous Improvement** | Model improves with more data |

---

### Real-World Example:

**Rule-Based Approach:**

## 6. Self-Check

- [x] Named ML task type: Binary Classification
- [x] Defined target: Client Churn (Yes/No)
- [x] Chosen success metric: F1-Score
- [x] Showed unit of analysis: One row = one client (aggregated)
- [x] Displayed real dataframe (client-level aggregation)
- [x] Explained why ML beats a fixed rule
- [x] Tied output to real action: Account managers contact high-risk clients

**✅ Ready to submit!**

---

### Summary:

| Aspect | What We Decided |
|--------|-----------------|
| **Task Type** | Binary Classification |
| **Target** | Client Churn (0 = Stayed, 1 = Churned) |
| **Metric** | F1-Score (balance of precision/recall) |
| **Unit** | One row = one client |
| **Why ML** | Complex, non-linear patterns across multiple features |

---

### Submission URL:
https://github.com/Hamna-Riaz0375/flyrankAI-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb